# 最高到達accuracyを取得する

In [30]:
from datetime import date

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import os
import json
import re
import sys
from collections import defaultdict

project_root = os.path.join(os.getcwd(), "..")
result_dir = os.path.join(project_root, "results")


def load_file_and_extract_max_accs(date, lr, target_concept_config_name, model_size, result_dir):
    """各initVecTypeとseedに対して、回したepochs中の最高到達accuracyを抽出する関数"""
    # if target_concept_config_type != '':
    #     target_concept_config_type = '-' + target_concept_config_type
    # dirname_pattern1 = f"gemma-3-{model_size}B-lr0.01{target_concept_config_type}-{date}-hidden_layer(-?\\d+)-seed(\\d+)_target_concepts_initvecwith(.*)"
    # dirname_pattern2 = f"gemma-3-{model_size}B-lr0.01{target_concept_config_type}-{date}-seed(\\d+)_target_concepts_initvecwith(.*)"

    target_concept_config_name = target_concept_config_name.replace('.json', '')
    dirname_pattern1 = f"gemma-3-{model_size}B-lr{lr}-{date}-hidden_layer(-?\\d+)-seed(\\d+)_{target_concept_config_name}_initvecwith(.*)"
    dirname_pattern2 = f"gemma-3-{model_size}B-lr{lr}-{date}-seed(\\d+)_{target_concept_config_name}_initvecwith(.*)"
    initVecType_to_seed_to_maxacc = defaultdict(dict)
    initVecType_to_seed_to_epoch_at_maxacc = defaultdict(dict)
    for dirname in os.listdir(result_dir):

        if re.match(dirname_pattern1, dirname):
            match_groups = re.search(dirname_pattern1, dirname)
            layer_idx = int(match_groups.group(1))
            seed = int(match_groups.group(2))
            initVecType = f"{match_groups.group(3)}_layer{layer_idx}"
        elif re.match(dirname_pattern2, dirname):
            match_groups = re.search(dirname_pattern2, dirname)
            print(match_groups)
            seed = int(match_groups.group(1))
            initVecType = match_groups.group(2)
        else:
            if date in dirname:
                # print(dirname)
                pass
            continue  # ディレクトリ名がどちらのパターンにもマッチしない場合はスキップ

        # print(f"Processing directory: {dirname}, initVecType: {initVecType}, seed: {seed}")
        
        if not os.path.exists(os.path.join(result_dir, dirname, 'logit_score_summary.json')):
            # print(f"Warning: logit_score_summary.json not found in {dirname}. Skipping.")
            continue
        with open(os.path.join(result_dir, dirname, 'logit_score_summary.json'), "r") as f:
            concept_to_epoch_to_scores = json.load(f)

        # 各conceptについて、回したepochs中の最高到達accuracyとそのepochを抽出
        max_accs = []
        epochs_at_maxacc = []
        for epoch_to_scores in concept_to_epoch_to_scores.values():
            max_accuracy = 0
            max_epoch_at_maxacc = 0
            for epoch, scores in epoch_to_scores.items():
                # print(f"Epoch: {epoch}, Accuracy: {scores['accuracy']}")
                if scores['accuracy'] > max_accuracy:
                    max_accuracy = scores['accuracy']
                    max_epoch_at_maxacc = epoch
            max_accs.append(max_accuracy)
            epochs_at_maxacc.append(max_epoch_at_maxacc)
        # print(f"Max Accuracies: {max_accs}")
        # print(f"Epochs at Max Accuracies: {epochs_at_maxacc}")
        initVecType_to_seed_to_maxacc[initVecType][seed] = max_accs
        initVecType_to_seed_to_epoch_at_maxacc[initVecType][seed] = epochs_at_maxacc
    return initVecType_to_seed_to_maxacc, initVecType_to_seed_to_epoch_at_maxacc


## 12B

In [109]:

date = '20260418'
model_size = '12'

target_concept_config_name = 'target_concepts_mini_13.json'


In [120]:
lr = 0.003
initVecType_to_seed_to_maxacc, initVecType_to_seed_to_epoch_at_maxacc = load_file_and_extract_max_accs(
    date, lr, target_concept_config_name, model_size, result_dir
)

for initVecType, seed_to_maxacc in initVecType_to_seed_to_maxacc.items():
    for seed, max_accs in seed_to_maxacc.items():
        mean_max_acc = np.mean(max_accs)
        std_max_acc = np.std(max_accs)
        print(f"InitVecType: {initVecType:15}, Seed: {seed}, Mean Max Accuracy: {mean_max_acc:.4f}, Std: {std_max_acc:.4f}")
print()
for initVecType, seed_to_epoch_at_maxacc in initVecType_to_seed_to_epoch_at_maxacc.items():
    for seed, epochs_at_maxacc in seed_to_epoch_at_maxacc.items():
        mean_epoch_at_maxacc = np.mean([int(e) for e in epochs_at_maxacc])
        std_epoch_at_maxacc = np.std([int(e) for e in epochs_at_maxacc])
        print(f"InitVecType: {initVecType:15}, Seed: {seed}, Mean Epoch number at Max Accuracy: {mean_epoch_at_maxacc:.2f}, Std: {std_epoch_at_maxacc:.2f}")


InitVecType: CatCent_by_WikiSummaryRepeatHSMixed_layer12, Seed: 12, Mean Max Accuracy: 0.6947, Std: 0.1488
InitVecType: CatCent_by_WikiSummaryRepeatHSMixed_layer12, Seed: 19, Mean Max Accuracy: 0.7142, Std: 0.1375
InitVecType: CatCent_by_WikiSummaryRepeatHSMixed_layer12, Seed: 11, Mean Max Accuracy: 0.7447, Std: 0.1353
InitVecType: CatCent_by_WikiSummaryRepeatHSMixed_layer12, Seed: 15, Mean Max Accuracy: 0.7357, Std: 0.1354
InitVecType: CatCent_by_WikiSummaryRepeatHSMixed_layer12, Seed: 18, Mean Max Accuracy: 0.7614, Std: 0.1470
InitVecType: CatCent_by_WikiSummaryRepeatHSMixed_layer12, Seed: 17, Mean Max Accuracy: 0.7069, Std: 0.1221
InitVecType: CatCent_by_WikiSummaryRepeatHSMixed_layer12, Seed: 14, Mean Max Accuracy: 0.7134, Std: 0.1433
InitVecType: CatCent_by_WikiSummaryRepeatHSMixed_layer12, Seed: 10, Mean Max Accuracy: 0.7066, Std: 0.1284
InitVecType: CatCent_by_WikiSummaryRepeatHSMixed_layer12, Seed: 13, Mean Max Accuracy: 0.7241, Std: 0.1517
InitVecType: CatCent_by_WikiSummaryRe

In [ ]:

initVecType_to_seed_to_maxacc, initVecType_to_seed_to_epoch_at_maxacc = load_file_and_extract_max_accs(
    date, lr, target_concept_config_name, model_size, result_dir
)
acc_cat = []
acc_other = []
for seed in range(10):
    if seed in initVecType_to_seed_to_maxacc["CatCent_by_WikiSummaryRepeatHSMixed_layer12"]:
        max_accs = initVecType_to_seed_to_maxacc["CatCent_by_WikiSummaryRepeatHSMixed_layer12"][seed]
        mean_max_acc = np.mean(max_accs)
        acc_cat.append(mean_max_acc)
    if seed in initVecType_to_seed_to_maxacc["otherCatCent_by_WikiSummaryRepeatHSMixed_layer12"]:
        max_accs = initVecType_to_seed_to_maxacc["otherCatCent_by_WikiSummaryRepeatHSMixed_layer12"][seed]
        mean_max_acc = np.mean(max_accs)
        acc_other.append(mean_max_acc)

In [ ]:

initVecType_to_seed_to_maxacc, initVecType_to_seed_to_epoch_at_maxacc = load_file_and_extract_max_accs(
    date, lr, target_concept_config_name, model_size, result_dir
)
acc_cat = []
acc_other = []
for seed in range(10):
    if seed in initVecType_to_seed_to_maxacc["CatCent_by_WikiSummaryRepeatHSMixed_layer12"]:
        max_accs = initVecType_to_seed_to_maxacc["CatCent_by_WikiSummaryRepeatHSMixed_layer12"][seed]
        mean_max_acc = np.mean(max_accs)
        acc_cat.append(mean_max_acc)
    if seed in initVecType_to_seed_to_maxacc["otherCatCent_by_WikiSummaryRepeatHSMixed_layer12"]:
        max_accs = initVecType_to_seed_to_maxacc["otherCatCent_by_WikiSummaryRepeatHSMixed_layer12"][seed]
        mean_max_acc = np.mean(max_accs)
        acc_other.append(mean_max_acc)

seed: 0, max_accs: [0.5121951219512195, 0.5147058823529411, 0.8768115942028986, 0.6, 0.6538461538461539, 0.9310344827586207, 0.5714285714285714, 0.9444444444444444, 0.875, 0.9473684210526315, 0.5882352941176471, 0.8421052631578947, 0.8333333333333334, 0.7383177570093458, 0.9487179487179487, 0.5957446808510638, 0.9019607843137255, 0.7857142857142857, 0.776536312849162, 0.8431372549019608, 0.3888888888888889, 0.4411764705882353, 0.845360824742268, 0.5806451612903226, 0.75, 0.6428571428571429], mean_max_acc: 0.728060233668104
seed: 1, max_accs: [0.6585365853658537, 0.7794117647058824, 0.8043478260869565, 0.5777777777777777, 0.6153846153846154, 0.6896551724137931, 0.6, 1.0, 0.55, 0.6842105263157895, 0.5588235294117647, 0.868421052631579, 0.8666666666666667, 0.719626168224299, 0.7948717948717948, 0.6808510638297872, 0.9019607843137255, 0.7285714285714285, 0.7262569832402235, 0.8431372549019608, 0.6111111111111112, 0.8529411764705882, 0.7319587628865979, 0.6129032258064516, 0.816666666666666

In [ ]:


acc_cat = []
acc_other = []


# ~9epoch Cat
date = '20260417'
initVecType_to_seed_to_maxacc, initVecType_to_seed_to_epoch_at_maxacc = load_file_and_extract_max_accs(
    date, lr, target_concept_config_name, model_size, result_dir
)
for seed in range(10):
    if seed in initVecType_to_seed_to_maxacc["CatCent_by_WikiSummaryRepeatHSMixed_layer12"]:
        max_accs = initVecType_to_seed_to_maxacc["CatCent_by_WikiSummaryRepeatHSMixed_layer12"][seed]
        mean_max_acc = np.mean(max_accs)
        acc_cat.append(mean_max_acc)
        print(f"seed: {seed}, max_accs: {max_accs}, mean_max_acc: {mean_max_acc}")
print(f"acc_cat: {acc_cat}")



# 10~ Cat, otherCat
date = '20260418'
initVecType_to_seed_to_maxacc, initVecType_to_seed_to_epoch_at_maxacc = load_file_and_extract_max_accs(
    date, lr, target_concept_config_name, model_size, result_dir
)
for seed in range(20):
    if seed < 10:        
        if seed in initVecType_to_seed_to_maxacc["otherCatCent_by_WikiSummaryRepeatHSMixed_layer12"]:
            max_accs = initVecType_to_seed_to_maxacc["otherCatCent_by_WikiSummaryRepeatHSMixed_layer12"][seed]
            mean_max_acc = np.mean(max_accs)
            acc_other.append(mean_max_acc)
            print(f"seed: {seed}, max_accs: {max_accs}, mean_max_acc: {mean_max_acc}")

    else:
        if seed in initVecType_to_seed_to_maxacc["CatCent_by_WikiSummaryRepeatHSMixed_layer12"]:
            max_accs = initVecType_to_seed_to_maxacc["CatCent_by_WikiSummaryRepeatHSMixed_layer12"][seed]
            mean_max_acc = np.mean(max_accs)
            acc_cat.append(mean_max_acc)
            print(f"seed: {seed}, max_accs: {max_accs}, mean_max_acc: {mean_max_acc}")
        if seed in initVecType_to_seed_to_maxacc["otherCatCent_by_WikiSummaryRepeatHSMixed_layer12"]:
            max_accs = initVecType_to_seed_to_maxacc["otherCatCent_by_WikiSummaryRepeatHSMixed_layer12"][seed]
            mean_max_acc = np.mean(max_accs)
            acc_other.append(mean_max_acc)
            print(f"seed: {seed}, max_accs: {max_accs}, mean_max_acc: {mean_max_acc}")

print(f"acc_other: {acc_other}")

In [108]:


acc_cat = []
acc_other = []


# ~9epoch Cat
date = '20260417'
initVecType_to_seed_to_maxacc1, initVecType_to_seed_to_epoch_at_maxacc = load_file_and_extract_max_accs(
    date, lr, target_concept_config_name, model_size, result_dir
)

# 10~ Cat, otherCat
date = '20260418'
initVecType_to_seed_to_maxacc2, initVecType_to_seed_to_epoch_at_maxacc = load_file_and_extract_max_accs(
    date, lr, target_concept_config_name, model_size, result_dir
)

for seed in range(20):
    if seed < 10:
        max_accs = initVecType_to_seed_to_maxacc1["CatCent_by_WikiSummaryRepeatHSMixed_layer12"][seed]
        mean_max_acc = np.mean(max_accs)
        acc_cat.append(mean_max_acc)
        print(f"seed: {seed}, max_accs: {max_accs}, mean_max_acc: {mean_max_acc}")

        max_accs = initVecType_to_seed_to_maxacc2["otherCatCent_by_WikiSummaryRepeatHSMixed_layer12"][seed]
        mean_max_acc = np.mean(max_accs)
        acc_other.append(mean_max_acc)
        print(f"seed: {seed}, max_accs: {max_accs}, mean_max_acc: {mean_max_acc}")
    
    else:
        max_accs = initVecType_to_seed_to_maxacc2["CatCent_by_WikiSummaryRepeatHSMixed_layer12"][seed]
        mean_max_acc = np.mean(max_accs)
        acc_cat.append(mean_max_acc)
        print(f"seed: {seed}, max_accs: {max_accs}, mean_max_acc: {mean_max_acc}")

        max_accs = initVecType_to_seed_to_maxacc2["otherCatCent_by_WikiSummaryRepeatHSMixed_layer12"][seed]
        mean_max_acc = np.mean(max_accs)
        acc_other.append(mean_max_acc)
        print(f"seed: {seed}, max_accs: {max_accs}, mean_max_acc: {mean_max_acc}")

print(f"acc_cat: {acc_cat}")
print(f"acc_other: {acc_other}")


seed: 0, max_accs: [0.5121951219512195, 0.5147058823529411, 0.8768115942028986, 0.6, 0.6538461538461539, 0.9310344827586207, 0.5714285714285714, 0.9444444444444444, 0.875, 0.9473684210526315, 0.5882352941176471, 0.8421052631578947, 0.8333333333333334, 0.7383177570093458, 0.9487179487179487, 0.5957446808510638, 0.9019607843137255, 0.7857142857142857, 0.776536312849162, 0.8431372549019608, 0.3888888888888889, 0.4411764705882353, 0.845360824742268, 0.5806451612903226, 0.75, 0.6428571428571429], mean_max_acc: 0.728060233668104
seed: 0, max_accs: [0.4634146341463415, 0.7205882352941176, 0.855072463768116, 0.4666666666666667, 0.5, 0.7931034482758621, 0.6, 0.7222222222222222, 0.55, 0.7894736842105263, 0.6176470588235294, 0.6842105263157895, 0.9, 0.7850467289719626, 0.7948717948717948, 0.6595744680851063, 0.8627450980392157, 0.7285714285714285, 0.8603351955307262, 0.7843137254901961, 0.7222222222222222, 0.5588235294117647, 0.8041237113402062, 0.6774193548387096, 0.9, 0.7857142857142857], mean_

In [102]:
import numpy as np
from scipy import stats

# 各seedのaccuracy（同じseed順で並べる）
acc_cat = np.array(acc_cat)
acc_other = np.array(acc_other)

# 差を取る
diff = acc_cat - acc_other

print("diff:", diff)
print("mean diff:", diff.mean())
print("std diff :", diff.std())

# paired t-test
t_stat, p_value = stats.ttest_1samp(diff, 0)

print("t =", t_stat)
print("p =", p_value)

diff: [ 0.01320791 -0.03898659  0.0502401   0.07699497 -0.02169269  0.00023097
  0.03236357 -0.00321076 -0.02135089 -0.01080597 -0.01537028 -0.02250985
 -0.00539513  0.00850081 -0.00605697 -0.04732564  0.02565926 -0.03785373
  0.05721165  0.02067759]
mean diff: 0.002726417591478936
std diff : 0.03241276460262966
t = 0.36665119143166436
p = 0.7179306191941247


### propnoun毎にmax accの差をとる

In [104]:


acc_cat = []
acc_other = []
acc_diffs_lst = []
acc_diff_means = []

# ~9epoch Cat
date = '20260417'
initVecType_to_seed_to_maxacc1, initVecType_to_seed_to_epoch_at_maxacc = load_file_and_extract_max_accs(
    date, lr, target_concept_config_name, model_size, result_dir
)

# 10~ Cat, otherCat
date = '20260418'
initVecType_to_seed_to_maxacc2, initVecType_to_seed_to_epoch_at_maxacc = load_file_and_extract_max_accs(
    date, lr, target_concept_config_name, model_size, result_dir
)

for seed in range(20):
    if seed < 10:
        max_accs = initVecType_to_seed_to_maxacc1["CatCent_by_WikiSummaryRepeatHSMixed_layer12"][seed]
        print(f"seed: {seed}, max_accs: {max_accs}")

        max_accs_other = initVecType_to_seed_to_maxacc2["otherCatCent_by_WikiSummaryRepeatHSMixed_layer12"][seed]
        print(f"seed: {seed}, max_accs_other: {max_accs_other}")

        acc_diffs = [max_acc - max_acc_other for max_acc, max_acc_other in zip(max_accs, max_accs_other)]
        # acc_diffs_lst.append(acc_diffs)
        acc_diff_mean = np.mean(acc_diffs)
        acc_diff_means.append(acc_diff_mean)
    else:
        max_accs = initVecType_to_seed_to_maxacc2["CatCent_by_WikiSummaryRepeatHSMixed_layer12"][seed]
        print(f"seed: {seed}, max_accs: {max_accs}")

        max_accs_other = initVecType_to_seed_to_maxacc2["otherCatCent_by_WikiSummaryRepeatHSMixed_layer12"][seed]
        print(f"seed: {seed}, max_accs_other: {max_accs_other}")

        acc_diffs = [max_acc - max_acc_other for max_acc, max_acc_other in zip(max_accs, max_accs_other)]
        # acc_diffs_lst.append(acc_diffs)
        acc_diff_mean = np.mean(acc_diffs)
        acc_diff_means.append(acc_diff_mean)

# print(f"acc_diffs_lst: {acc_diffs_lst}")
print(f"acc_diff_means: {acc_diff_means}")


seed: 0, max_accs: [0.5121951219512195, 0.5147058823529411, 0.8768115942028986, 0.6, 0.6538461538461539, 0.9310344827586207, 0.5714285714285714, 0.9444444444444444, 0.875, 0.9473684210526315, 0.5882352941176471, 0.8421052631578947, 0.8333333333333334, 0.7383177570093458, 0.9487179487179487, 0.5957446808510638, 0.9019607843137255, 0.7857142857142857, 0.776536312849162, 0.8431372549019608, 0.3888888888888889, 0.4411764705882353, 0.845360824742268, 0.5806451612903226, 0.75, 0.6428571428571429]
seed: 0, max_accs_other: [0.4634146341463415, 0.7205882352941176, 0.855072463768116, 0.4666666666666667, 0.5, 0.7931034482758621, 0.6, 0.7222222222222222, 0.55, 0.7894736842105263, 0.6176470588235294, 0.6842105263157895, 0.9, 0.7850467289719626, 0.7948717948717948, 0.6595744680851063, 0.8627450980392157, 0.7285714285714285, 0.8603351955307262, 0.7843137254901961, 0.7222222222222222, 0.5588235294117647, 0.8041237113402062, 0.6774193548387096, 0.9, 0.7857142857142857]
seed: 1, max_accs: [0.65853658536

In [105]:
acc_diff_means

[np.float64(0.013207907406150611),
 np.float64(-0.038986585599968995),
 np.float64(0.050240097245513),
 np.float64(0.07699497423179445),
 np.float64(-0.02169269211743766),
 np.float64(0.00023097477839071644),
 np.float64(0.03236357221115401),
 np.float64(-0.0032107610064951034),
 np.float64(-0.021350892397995883),
 np.float64(-0.01080596839365389),
 np.float64(-0.015370277975169093),
 np.float64(-0.022509847430528793),
 np.float64(-0.005395134211683526),
 np.float64(0.008500813622198292),
 np.float64(-0.006056965033106765),
 np.float64(-0.04732564267603405),
 np.float64(0.025659262600865738),
 np.float64(-0.03785372623082802),
 np.float64(0.057211652702281984),
 np.float64(0.020677590104132025)]

In [107]:
import numpy as np
from scipy import stats


print("diff:", acc_diff_means)
print("mean diff:", np.mean(acc_diff_means))
print("std diff :", np.std(acc_diff_means))

# paired t-test
t_stat, p_value = stats.ttest_1samp(acc_diff_means, 0)

print("t =", t_stat)
print("p =", p_value)

diff: [np.float64(0.013207907406150611), np.float64(-0.038986585599968995), np.float64(0.050240097245513), np.float64(0.07699497423179445), np.float64(-0.02169269211743766), np.float64(0.00023097477839071644), np.float64(0.03236357221115401), np.float64(-0.0032107610064951034), np.float64(-0.021350892397995883), np.float64(-0.01080596839365389), np.float64(-0.015370277975169093), np.float64(-0.022509847430528793), np.float64(-0.005395134211683526), np.float64(0.008500813622198292), np.float64(-0.006056965033106765), np.float64(-0.04732564267603405), np.float64(0.025659262600865738), np.float64(-0.03785372623082802), np.float64(0.057211652702281984), np.float64(0.020677590104132025)]
mean diff: 0.0027264175914789536
std diff : 0.0324127646026297
t = 0.3666511914316663
p = 0.7179306191941232


## epoch4固定でaccuracyを計算する

In [87]:
sys.path.append(project_root)

from utils.gemma_train_and_test_utils import calculate_metrics

def get_concept_to_epoch_score(concept_result_dir, epoch):
    concept_to_results = {}
    for dirname in os.listdir(concept_result_dir):
        if dirname.endswith(".json"):
            continue
        concept = dirname

        # このconceptに対するepochの結果を読み込む
        logit_score_path = os.path.join(concept_result_dir, dirname, f"logit-scored_epoch{epoch}.json")
        
        if not os.path.exists(logit_score_path):
            # raise FileNotFoundError(f"Logit score file not found: {logit_score_path}")
            # print(f"Logit score file not found: {logit_score_path}. Skipping this concept.")
            continue
            
        with open(logit_score_path, 'r') as f:
            results = json.load(f)
        y_true_lst = [res['label'] for res in results]
        y_pred_lst = [res['pred'] for res in results]

        # score計算
        metrics = calculate_metrics(y_pred_lst, y_true_lst)

        concept_to_results[concept] = metrics

        # print(f"  Epoch {epoch}: Accuracy: {metrics['accuracy']:.4f}, Precision: {metrics['precision']:.4f}, Recall: {metrics['recall']:.4f}, F1: {metrics['F1']:.4f}")
    
    return concept_to_results





def load_file_and_calc_epoch_accs(date, epoch, lr, target_concept_config_name, model_size, result_dir, print_flag=False):
    """各initVecTypeとseedに対して、回したepochs中の最高到達accuracyを抽出する関数"""

    target_concept_config_name = target_concept_config_name.replace('.json', '')
    dirname_pattern1 = f"gemma-3-{model_size}B-lr{lr}-{date}-hidden_layer(-?\\d+)-seed(\\d+)_{target_concept_config_name}_initvecwith(.*)"
    dirname_pattern2 = f"gemma-3-{model_size}B-lr{lr}-{date}-seed(\\d+)_{target_concept_config_name}_initvecwith(.*)"

    initvectype_to_seed_to_acc = defaultdict(dict)
    for dirname in os.listdir(result_dir):

        if re.match(dirname_pattern1, dirname):
            match_groups = re.search(dirname_pattern1, dirname)
            layer_idx = int(match_groups.group(1))
            seed = int(match_groups.group(2))
            initVecType = f"{match_groups.group(3)}_layer{layer_idx}"
        elif re.match(dirname_pattern2, dirname):
            match_groups = re.search(dirname_pattern2, dirname)
            print(match_groups)
            seed = int(match_groups.group(1))
            initVecType = match_groups.group(2)
        else:
            if date in dirname:
                pass
            continue  # ディレクトリ名がどちらのパターンにもマッチしない場合はスキップ

        concept_to_results = get_concept_to_epoch_score(os.path.join(result_dir, dirname), epoch)
        if len(concept_to_results) == 0:
            # print(f"No valid concepts with logit score files found for directory: {dirname}. Skipping.")
            continue
        sum_acc = 0
        for concept, score in concept_to_results.items():
            acc = score['accuracy']
            sum_acc += acc

        avg_acc = sum_acc / len(concept_to_results)
        initvectype_to_seed_to_acc[initVecType][seed] = avg_acc
        if print_flag:
            print(f"InitVecType: {initVecType}, Seed: {seed}, Average Accuracy at Epoch {epoch}: {avg_acc:.4f}")

    return initvectype_to_seed_to_acc


In [92]:
# epoch = 2

# date = '20260417'

for epoch in range(1, 5):

    # ~9epoch Cat
    date = '20260417'
    initvectype_to_seed_to_acc1 = load_file_and_calc_epoch_accs(date, epoch, lr=lr, 
                                                            target_concept_config_name=target_concept_config_name, 
                                                            model_size=model_size, 
                                                            result_dir=result_dir,
                                                            print_flag=False)

    # 10~ Cat, otherCat
    date = '20260418'
    initvectype_to_seed_to_acc2 = load_file_and_calc_epoch_accs(date, epoch, lr=lr, 
                                                            target_concept_config_name=target_concept_config_name, 
                                                            model_size=model_size, 
                                                            result_dir=result_dir,
                                                            print_flag=False)


    # for seed in range(10):
    # ** Cat **
    acc_lst = []
    for seed in range(20):   
        if seed < 10: 
            # date = '20260417'
            initvectype_to_seed_to_acc = initvectype_to_seed_to_acc1
        else:
            # date = '20260418'
            initvectype_to_seed_to_acc = initvectype_to_seed_to_acc2
        for initVecType, seed_to_acc in initvectype_to_seed_to_acc.items():
            if initVecType == 'CatCent_by_WikiSummaryRepeatHSMixed_layer12':
                acc = seed_to_acc.get(seed, None)
                if acc is not None:
                    acc_lst.append(acc)
                    print(f"InitVecType: {initVecType}, Seed: {seed}, Average Accuracy at Epoch {epoch}: {acc:.4f}")
                else:
                    print(f"  InitVecType: {initVecType}, No data for Seed {seed}")
        
    # mean ± std
    mean = np.mean(acc_lst)
    std = np.std(acc_lst)
    print(f"accuracy at epoch {epoch}: {mean:.4f} ± {std:.4f}")
    print()
    cat_acc_lst = acc_lst


    # ** otherCat **
    acc_lst = []
    for seed in range(20):
        # date = '20260418'
        for initVecType, seed_to_acc in initvectype_to_seed_to_acc2.items():
            if initVecType == 'otherCatCent_by_WikiSummaryRepeatHSMixed_layer12':
                acc = seed_to_acc.get(seed, None)
                if acc is not None:
                    acc_lst.append(acc)
                    print(f"InitVecType: {initVecType}, Seed: {seed}, Average Accuracy at Epoch {epoch}: {acc:.4f}")
                else:
                    print(f"  InitVecType: {initVecType}, No data for Seed {seed}")

    # mean ± std
    mean = np.mean(acc_lst)
    std = np.std(acc_lst)
    print(f"accuracy at epoch {epoch}: {mean:.4f} ± {std:.4f}")
    othercat_acc_lst = acc_lst


    diff_lst = [cat_acc - othercat_acc for cat_acc, othercat_acc in zip(cat_acc_lst, othercat_acc_lst)]
    print(f"plus diff: {len([d for d in diff_lst if d > 0])} / {len(diff_lst)}")
    print(f"minus diff: {len([d for d in diff_lst if d < 0])} / {len(diff_lst)}")
    print(diff_lst)
    print('\n==============================\n')
    

InitVecType: CatCent_by_WikiSummaryRepeatHSMixed_layer12, Seed: 0, Average Accuracy at Epoch 1: 0.6257
InitVecType: CatCent_by_WikiSummaryRepeatHSMixed_layer12, Seed: 1, Average Accuracy at Epoch 1: 0.6551
InitVecType: CatCent_by_WikiSummaryRepeatHSMixed_layer12, Seed: 2, Average Accuracy at Epoch 1: 0.6543
InitVecType: CatCent_by_WikiSummaryRepeatHSMixed_layer12, Seed: 3, Average Accuracy at Epoch 1: 0.6498
InitVecType: CatCent_by_WikiSummaryRepeatHSMixed_layer12, Seed: 4, Average Accuracy at Epoch 1: 0.6056
InitVecType: CatCent_by_WikiSummaryRepeatHSMixed_layer12, Seed: 5, Average Accuracy at Epoch 1: 0.6399
InitVecType: CatCent_by_WikiSummaryRepeatHSMixed_layer12, Seed: 6, Average Accuracy at Epoch 1: 0.6424
InitVecType: CatCent_by_WikiSummaryRepeatHSMixed_layer12, Seed: 7, Average Accuracy at Epoch 1: 0.6271
InitVecType: CatCent_by_WikiSummaryRepeatHSMixed_layer12, Seed: 8, Average Accuracy at Epoch 1: 0.6047
InitVecType: CatCent_by_WikiSummaryRepeatHSMixed_layer12, Seed: 9, Averag

In [ ]:
import wandb

api = wandb.Api(timeout=60)
entity = "toko-tohoku-university"

projects = api.projects(entity=entity)

for project in projects:
    runs = api.runs(
        f"{entity}/{project.name}",
        filters={"created_at": {"$lt": "2025-09-01"}},
        per_page=50  # 安定性向上
    )
    
    for run in runs:
        print(f"Delete target: {entity}/{project.name}/{run.id} ({run.created_at})")
        # run.delete()

In [74]:
import wandb
from datetime import datetime, timezone

api = wandb.Api(timeout=60)

# https://wandb.ai/toko-tohoku-university/gemma-3-12B-lr0.003-20260418-hidden_layer12-seed11_target_concepts_mini_13_initvecwithCatCent_by_WikiSummaryRepeatHSMixed_2/runs/2gii366h
# 削除したい基準日時
# cutoff = datetime(2025, 9, 1)
cutoff = datetime(2025, 9, 1, tzinfo=timezone.utc)


entity = "toko-tohoku-university"  # or team名


# 全プロジェクト取得
projects = api.projects(entity=entity)

for project in projects:
    # print(f"Checking project: {project.name}")
    
    # runs = api.runs(f"{entity}/{project.name}")
    runs = api.runs(
        f"{entity}/{project.name}",
        filters={"created_at": {"$lt": "2025-09-01"}}
    )
    
    for run in runs:
        # 文字列 → datetime に変換
        created_at = datetime.fromisoformat(run.created_at.replace("Z", "+00:00"))
        created_at = created_at.replace(tzinfo=None)
        
        # print(f"Delete target: {entity}/{project.name}/{run.id} ({created_at})")
        if created_at < cutoff:
            # print(f"Delete target: {entity}/{project.name}/{run.id} ({created_at})")
            print(f"Deleting: {entity}/{project.name}/{run.id} ({created_at})")
            run.delete()